In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

2025-10-24 12:01:46.153676: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-24 12:01:46.559611: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-24 12:01:56.233738: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Configure GPU for maximum performance
def configure_gpu():
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            # Limit memory use to 20 out of 24GB
            tf.config.experimental.set_virtual_device_configuration(
                gpus[0],
                [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=20480)]  # 20GB
            )

            # Float32 precision as middle ground between accuracy and performance
            keras.mixed_precision.set_global_policy('float32')

            print(f"GPU configured: {gpus[0]}")

        except RuntimeError as e:
            print(f"GPU configuration error: {e}")
    else:
        print("No GPU found, stopping program...")
        exit()


# Load and prepare the dataset with GPU-optimized preprocessing
def load_imdb_data():
    # Load the dataset
    df = pd.read_csv('IMDB Dataset.csv')

    # Convert sentiments to binary labels
    df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

    texts = df['review'].values
    labels = df['sentiment'].values.astype(np.float32)  # Use float32 for GPU

    return texts, labels


# Character-level tokenization with GPU-friendly encoding
def create_char_encoder():
    # Create character vocabulary (256 characters for 8-bit representation)
    chars = [chr(i) for i in range(32, 127)]  # Printable ASCII characters
    chars.extend(['<PAD>', '<UNK>', '<START>', '<END>'])  # Special tokens

    char_to_id = {char: i for i, char in enumerate(chars)}
    id_to_char = {i: char for i, char in enumerate(chars)}

    return char_to_id, id_to_char, len(chars)


# Preprocess text to character sequences with GPU optimization
def text_to_char_sequence(text, char_to_id, max_length=1024):  # Increased sequence length for GPU
    text = text.lower()[:max_length - 2]  # Reserve space for start/end tokens

    # Add start and end tokens
    sequence = [char_to_id['<START>']]
    sequence.extend([char_to_id.get(char, char_to_id['<UNK>']) for char in text])
    sequence.append(char_to_id['<END>'])

    # Pad or truncate to max_length
    if len(sequence) < max_length:
        sequence = sequence + [char_to_id['<PAD>']] * (max_length - len(sequence))
    else:
        sequence = sequence[:max_length]
        sequence[-1] = char_to_id['<END>']  # Ensure end token at the end

    return sequence


# Batch preprocessing for GPU efficiency
def preprocess_dataset(texts, labels, char_to_id, max_length=1024, batch_size=32):
    char_sequences = [text_to_char_sequence(text, char_to_id, max_length) for text in texts]

    # Convert to TensorFlow dataset for GPU optimization
    dataset = tf.data.Dataset.from_tensor_slices((
        np.array(char_sequences, dtype=np.int32),
        np.array(labels, dtype=np.float32)
    ))

    # Optimize dataset performance
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    dataset = dataset.cache()  # Cache the preprocessed data

    return dataset


# GPU-optimized transformer components
def positional_encoding(length, depth):
    depth = depth / 2

    positions = tf.range(length, dtype=tf.float32)[:, tf.newaxis]  # (seq, 1)
    depths = tf.range(depth, dtype=tf.float32)[tf.newaxis, :] / depth  # (1, depth)

    angle_rates = 1.0 / tf.pow(10000.0, depths)  # (1, depth)
    angle_rads = positions * angle_rates  # (pos, depth)

    pos_encoding = tf.concat([tf.sin(angle_rads), tf.cos(angle_rads)], axis=-1)

    return tf.cast(pos_encoding, dtype=tf.float32)


class PositionalEmbedding(layers.Layer):
    def __init__(self, vocab_size, d_model, max_length=1024):
        super().__init__()
        self.d_model = d_model
        self.embedding = layers.Embedding(vocab_size, d_model, mask_zero=True)
        self.pos_encoding = positional_encoding(max_length, d_model)

    def compute_mask(self, *args, **kwargs):
        return self.embedding.compute_mask(*args, **kwargs)

    def call(self, x):
        length = tf.shape(x)[1]
        x = self.embedding(x)
        # Scale embeddings and add positional encoding
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = x + self.pos_encoding[tf.newaxis, :length, :]
        return x


class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()

        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout_rate
        )
        self.ffn = keras.Sequential([
            layers.Dense(dff, activation='relu'),
            layers.Dense(d_model)
        ])

        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        # Multi-head attention with residual connection
        attn_output = self.mha(x, x, x)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)

        # Feed forward network with residual connection
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)

        return out2


# Build larger model to utilize GPU memory
def build_gpu_optimized_transformer(vocab_size, max_length=1024):
    # Larger model parameters to utilize 3090's 24GB VRAM
    d_model = 256  # Increased from 256 to use more GPU memory
    num_heads = 8  # Increased attention heads
    dff = 512  # Increased feed-forward dimension
    num_layers = 6  # More transformer blocks
    dropout_rate = 0.1

    # Input layer
    inputs = layers.Input(shape=(max_length,), dtype=tf.int32, name='input_layer')

    # Character embeddings with positional encoding
    embedding = PositionalEmbedding(vocab_size, d_model, max_length)(inputs)

    # Transformer blocks
    x = TransformerBlock(d_model, num_heads, dff, dropout_rate)(embedding)
    x = TransformerBlock(d_model, num_heads, dff, dropout_rate)(x)

    # Global average pooling and classification
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(512, activation='relu')(x)  # Larger dense layer
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(128, activation='relu')(x)

    # Output layer with float32 for stability
    outputs = layers.Dense(1, activation='sigmoid', dtype='float32', name='output_layer')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

    return model


# Custom callback to monitor GPU usage
class GPUMonitorCallback(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        # Print GPU memory usage
        if tf.config.list_physical_devices('GPU'):
            gpu_stats = tf.config.experimental.get_memory_info('GPU:0')
            print(f"GPU Memory - Current: {gpu_stats['current'] / 1e9:.2f}GB, "
                  f"Peak: {gpu_stats['peak'] / 1e9:.2f}GB")


# Main training script optimized for GPU
def main():
    # Configure GPU first
    configure_gpu()

    # Load data
    print("Loading IMDB dataset...")
    texts, labels = load_imdb_data()

    # Create character encoder
    print("Creating character encoder...")
    char_to_id, id_to_char, vocab_size = create_char_encoder()
    print(f"Vocabulary size: {vocab_size}")

    # Use larger batch size and sequence length for GPU
    batch_size = 32  # Increased batch size for GPU
    max_length = 1024  # Longer sequences for better context

    # Create optimized datasets
    print("Creating GPU-optimized datasets...")
    X_train, X_test, y_train, y_test = train_test_split(
        texts, labels, test_size=0.5, shuffle=False  # 50/50 training and test/validation data split without shuffling
    )

    # Create TensorFlow datasets
    train_dataset = preprocess_dataset(X_train, y_train, char_to_id, max_length, batch_size)
    test_dataset = preprocess_dataset(X_test, y_test, char_to_id, max_length, batch_size)

    # Build larger model to utilize GPU
    print("Building GPU-optimized model...")
    model = build_gpu_optimized_transformer(vocab_size, max_length)

    # Compile with GPU-optimized settings
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=1e-4,
            beta_1=0.9,
            beta_2=0.98,
            epsilon=1e-9
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # Print model summary
    model.summary()

    # Callbacks for GPU training
    callbacks = [
        GPUMonitorCallback(),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=5, restore_best_weights=True
        ),
        keras.callbacks.ModelCheckpoint(
            'best_gpu_transformer.h5',
            monitor='val_accuracy',
            save_best_only=True,
            mode='max'
        )
    ]

    # Train model with GPU optimization
    print("Starting GPU-optimized training...")
    history = model.fit(
        train_dataset,
        epochs=20,  # More epochs since we have GPU power
        validation_data=test_dataset,
        callbacks=callbacks,
        verbose=1
    )

    # Evaluate final model
    print("Evaluating model...")
    test_loss, test_acc = model.evaluate(test_dataset, verbose=0)
    print(f"Final Test accuracy: {test_acc:.4f}")

    # Save the final model
    model.save('gpu_char_transformer_imdb.keras')
    print("GPU-optimized model saved!")


# GPU-optimized prediction function
def predict_sentiment_gpu(model, text, char_to_id, max_length=1024):
    # Preprocess on CPU, predict on GPU
    sequence = text_to_char_sequence(text, char_to_id, max_length)
    sequence = np.array([sequence], dtype=np.int32)

    # Convert to tensor and predict
    sequence_tensor = tf.convert_to_tensor(sequence)
    prediction = model.predict(sequence_tensor, verbose=0)[0][0]

    sentiment = "positive" if prediction > 0.5 else "negative"

    return sentiment, float(prediction)


if __name__ == "__main__":
    # Verify GPU is available
    print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
    main()

2025-10-24 12:28:41.909491: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-24 12:28:42.381052: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-24 12:28:50.836297: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Num GPUs Available:  1
GPU configured: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
Loading IMDB dataset...
Creating character encoder...
Vocabulary size: 99
Creating GPU-optimized datasets...


I0000 00:00:1761301738.215909   17801 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20480 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:01:00.0, compute capability: 8.6


Building GPU-optimized model...


/mnt/c/Users/Henrik/ml2/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'transformer_block' (of type TransformerBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


...

782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.5904 - loss: 0.6669 - val_accuracy: 0.5863 - val_loss: 0.6739 - learning_rate: 5.0000e-05
GPU Memory - Current: 0.02GB, Peak: 0.66GB
Epoch 9/20
780/782 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5900 - loss: 0.6668

782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - accuracy: 0.5922 - loss: 0.6660 - val_accuracy: 0.5878 - val_loss: 0.6744 - learning_rate: 5.0000e-05
GPU Memory - Current: 0.02GB, Peak: 0.66GB
Epoch 10/20
780/782 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5946 - loss: 0.6654

782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - accuracy: 0.5968 - loss: 0.6648 - val_accuracy: 0.5888 - val_loss: 0.6729 - learning_rate: 5.0000e-05
GPU Memory - Current: 0.02GB, Peak: 0.66GB
Epoch 11/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.5986 - loss: 0.6631 - val_accuracy: 0.5877 - val_loss: 0.6734 - learning_rate: 5.0000e-05
GPU Memory - Current: 0.02GB, Peak: 0.66GB
Epoch 12/20
778/782 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5977 - loss: 0.6625

782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.5987 - loss: 0.6621 - val_accuracy: 0.5901 - val_loss: 0.6740 - learning_rate: 5.0000e-05
GPU Memory - Current: 0.02GB, Peak: 0.66GB
Epoch 13/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.6010 - loss: 0.6609 - val_accuracy: 0.5872 - val_loss: 0.6744 - learning_rate: 5.0000e-05
GPU Memory - Current: 0.02GB, Peak: 0.66GB
Epoch 14/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.6069 - loss: 0.6582 - val_accuracy: 0.5871 - val_loss: 0.6748 - learning_rate: 2.5000e-05
GPU Memory - Current: 0.02GB, Peak: 0.66GB
Epoch 15/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.6080 - loss: 0.6574 - val_accuracy: 0.5874 - val_loss: 0.6748 - learning_rate: 2.5000e-05
GPU Memory - Current: 0.02GB, Peak: 0.66GB
Epoch 16/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.6085 - loss: 0.6561 - val_accuracy: 0.5884 - val_loss: 0.6753 - learning_rate: 2.5000e-05
GPU Memory - Current: 0.02GB, Peak: 0.66GB
Ep

Final Test accuracy: 0.5901
GPU-optimized model saved!
